# Fase B · Pipeline ETL
Grupo 1 | PostgreSQL + SQLAlchemy

Ejecutar con **Restart Kernel & Run All Cells**. Configurar primero `.env` según el README.

> **Alcance:** análisis del archivo recibido. Todas sus fechas tienen día entre 1 y 12; no se conoce su cobertura completa ni se ha confirmado el CSV oficial. Se conservan filas idénticas por falta de un identificador de evento. `CANTIDAD` se suma; no equivale necesariamente al número de filas.

In [1]:
from pathlib import Path
import sys
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/proyecto.py').is_file()), None)
if RAIZ is None:
    raise RuntimeError('Abra el notebook desde la carpeta del proyecto o entrega/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
import pandas as pd
from IPython.display import display, Markdown
get_ipython().run_line_magic('matplotlib', 'inline')
from src.proyecto import extraer, transformar, conectar, cargar, consultar


## 1. Extracción
El CSV incluido es una conversión de formato del Excel aportado. `scripts/preparar_fuente.py` documenta la operación; no se afirma que sea el CSV oficial exigido por la rúbrica. Se leen los campos como texto para conservar códigos y detectar conversiones inválidas.

In [2]:
fuente, huella = extraer()
display(fuente.head())
display(pd.DataFrame({"tipo_original_csv": fuente.dtypes.astype(str)}))
print(f"Filas: {len(fuente):,}; columnas: {len(fuente.columns)}; SHA-256: {huella}")

,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,ZONA,SEXO,ARMA MEDIO,MODALIDAD PRESUNTA,SPOA_CARACTERIZACION,CANTIDAD
0,2025-12-12,8,ATLANTICO,8433,MALAMBO,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1
1,2025-12-12,8,ATLANTICO,8573,PUERTO COLOMBIA,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1
2,2025-12-12,8,ATLANTICO,8638,SABANALARGA,URBANA,FEMENINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1
3,2025-12-12,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,FEMENINO,ARMA BLANCA / CORTOPUNZANTE,POR ESTABLECER,HOMICIDIO INTENCIONAL O CON DOLO,1
4,2025-12-12,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1


,tipo_original_csv
FECHA HECHO,string
COD_DEPTO,string
DEPARTAMENTO,string
COD_MUNI,string
MUNICIPIO,string
ZONA,string
SEXO,string
ARMA MEDIO,string
MODALIDAD PRESUNTA,string
SPOA_CARACTERIZACION,string


Filas: 31,708; columnas: 11; SHA-256: 5a068e689711c1750e76788e41f94304061992b7233a5b49bd7b54bc3e0e5641


## 2. Transformación
Se normalizan espacios, mayúsculas y Unicode; se convierten fechas ISO y cantidades enteras positivas. Los códigos se conservan como texto con ceros iniciales. Un municipio vacío se completa únicamente desde una correspondencia unívoca del mismo código dentro de la fuente.

Se construyen cuatro dimensiones con claves subrogadas y un hecho por fila de origen. Las filas repetidas se conservan; los merges son `many_to_one`. El código completo y comentado está en `src/proyecto.py`.

In [3]:
tablas, datos, auditoria = transformar(fuente, huella)
display(pd.DataFrame([{"tabla": nombre, "filas": len(tabla)} for nombre, tabla in tablas.items()]))
display(auditoria)
for nombre, tabla in tablas.items():
    display(Markdown(f"### {nombre}"))
    display(tabla.head(3))

,tabla,filas
0,dim_tiempo,864
1,dim_ubicacion,1565
2,dim_sexo,3
3,dim_contexto,84
4,hecho_homicidios,31708


{'filas_fuente': 31708,
 'filas_hecho': 31708,
 'suma_cantidad': 31746,
 'duplicados_exactos_conservados': 4383,
 'nombres_completados_por_codigo': 1,
 'textos_vacios_originales': {'departamento': 0,
  'municipio': 1,
  'zona': 0,
  'sexo': 0,
  'arma_medio': 0,
  'modalidad_presunta': 0,
  'caracterizacion': 0},
 'fecha_min': '2020-01-01',
 'fecha_max': '2025-12-12',
 'dias_distintos': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
 'fuente_sha256': '5a068e689711c1750e76788e41f94304061992b7233a5b49bd7b54bc3e0e5641'}

### dim_tiempo

,tiempo_id,fecha,anio,mes,dia,trimestre
0,1,2020-01-01,2020,1,1,1
1,2,2020-01-02,2020,1,2,1
2,3,2020-01-03,2020,1,3,1


### dim_ubicacion

...

,sexo_id,sexo
0,1,FEMENINO
1,2,MASCULINO
2,3,SIN ESTABLECER


### dim_contexto

,contexto_id,arma_medio,modalidad_presunta,caracterizacion
0,1,ARMA BLANCA / CORTOPUNZANTE,ATRACO,HOMICIDIO INTENCIONAL O CON DOLO
1,2,ARMA BLANCA / CORTOPUNZANTE,DESMEMBRAMIENTO,HOMICIDIO INTENCIONAL O CON DOLO
2,3,ARMA BLANCA / CORTOPUNZANTE,POR ESTABLECER,FEMINICIDIO


### hecho_homicidios

,hecho_id,fuente_sha256,fila_fuente,tiempo_id,ubicacion_id,sexo_id,contexto_id,cantidad
0,1,5a068e689711c1750e76788e41f94304061992b7233a5b...,2,864,237,2,49,1
1,2,5a068e689711c1750e76788e41f94304061992b7233a5b...,3,864,247,2,49,1
2,3,5a068e689711c1750e76788e41f94304061992b7233a5b...,4,864,253,1,49,1


## 3. Validaciones antes de cargar
Se verifican conservación de filas y de la suma, unicidad de claves, ausencia de nulos y pertenencia de cada FK a su dimensión. Los días 13 a 31 ausentes se reportan como alerta, sin imputarlos como cero.

In [4]:
from src.proyecto import DIMENSIONES
hecho = tablas['hecho_homicidios']
assert len(hecho) == len(fuente)
assert int(hecho.cantidad.sum()) == int(pd.to_numeric(fuente.CANTIDAD).sum())
assert hecho.hecho_id.is_unique
assert not hecho.isna().any().any()
for nombre, (clave, _) in DIMENSIONES.items():
    assert tablas[nombre][clave].is_unique
    assert hecho[clave].isin(tablas[nombre][clave]).all()
print('Validaciones de integridad superadas.')
if max(auditoria['dias_distintos']) <= 12:
    print('ALERTA: solo aparecen los días 1 a 12. Confirmar cobertura y conversión con la fuente.')

Validaciones de integridad superadas.
ALERTA: solo aparecen los días 1 a 12. Confirmar cobertura y conversión con la fuente.


## 4. Carga a PostgreSQL
Se usa `DATABASE_URL` de `.env`. La recarga afecta únicamente las cinco tablas de `p1_homicidios`, esquema reservado a este proyecto. La transacción mantiene PK, FK, UNIQUE y CHECK; cualquier error revierte la carga. Repetir el notebook no acumula filas.

In [5]:
motor = conectar()
control = cargar(motor, tablas)
display(control)

{'filas_postgresql': 31708, 'cantidad_postgresql': 31746}

## 5. Comprobación en la base
El control siguiente se consulta directamente en PostgreSQL.

In [6]:
display(consultar(motor, """SELECT COUNT(*) AS filas, SUM(cantidad) AS cantidad,
COUNT(DISTINCT fuente_sha256) AS fuentes FROM p1_homicidios.hecho_homicidios"""))
motor.dispose()

,filas,cantidad,fuentes
0,31708,31746,1


## Reflexión: carga incremental
La recarga completa resulta razonable para 31.708 filas. Para incrementos se requiere un identificador estable del evento y un criterio de actualización de la fuente: la pareja hash de archivo + fila evita duplicar una misma carga, pero no reconoce eventos repetidos en archivos nuevos.

Se usaría staging, búsqueda de claves dimensionales existentes y `INSERT ... ON CONFLICT` sobre una clave de negocio validada. Las claves subrogadas actuales se regeneran de forma determinista para la misma fuente; no son persistentes ante cambios del conjunto. No se deben aplicar como estrategia incremental sin adaptar el diseño.